### Imports

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from src.wti_utils import build_wti_ohlc_features, create_labels


### CSV-Daten laden

minütliche WTI Future Intraday Kurse in USD von 2011-2022

In [2]:
wti = pd.read_csv(r"C:/Users/02_Florian_Benutzer/Desktop/trump_oil_analysis_v2/00_data\wti.csv")

In [3]:
wti_v1 = wti.copy()

Zeitzone des Datensatzes bestimmen, da auf Kaggle nicht angegebem

In [4]:
# Einbruch der counts um 17 Uhr deutet auf NY Zeitzone hin

# Mit diesem Python-Code gruppierst du die Daten nach der Uhrzeit, die aktuell im Datensatz steht, und zählst, wie viele Einträge es pro Stunde gibt:

# Fall 1: Die Lücke ist bei 17 Uhr. -> Deine Daten sind bereits in der New York Zeitzone (EST/EDT).

wti_v1['time'] = pd.to_datetime(wti_v1['time'])

# Stunde extrahieren (0 bis 23)
wti_v1['hour'] = wti_v1['time'].dt.hour

# Anzahl der Datenpunkte pro Stunde zählen
print(wti_v1['hour'].value_counts().sort_index())

hour
0     145365
1     154919
2     171426
3     182765
4     183397
5     182139
6     181676
7     182371
8     184237
9     185107
10    185036
11    185006
12    184443
13    180690
14    176731
15    167613
16    147228
17     19011
18    134967
19    129162
20    158876
21    161296
22    153402
23    146162
Name: count, dtype: int64


In [5]:
# Nur Freitage filtern - da Börsenschluss im 17 Uhr - deutet auf NY Zeit hin

# schauen wir uns den letzten Datenpunkt vor dem Wochenende an. Freitags um 17:00 Uhr New York

freitage = wti_v1[wti_v1['time'].dt.dayofweek == 4]

# Die jeweils letzte Uhrzeit an jedem Freitag anzeigen
print(freitage.groupby(freitage['time'].dt.date)['time'].max().head(10))

time
2011-01-07   2011-01-07 17:00:00
2011-01-14   2011-01-14 16:59:00
2011-01-21   2011-01-21 17:00:00
2011-01-28   2011-01-28 16:59:00
2011-02-04   2011-02-04 16:59:00
2011-02-11   2011-02-11 16:59:00
2011-02-18   2011-02-18 16:59:00
2011-02-25   2011-02-25 16:59:00
2011-03-04   2011-03-04 16:59:00
2011-03-11   2011-03-11 16:59:00
Name: time, dtype: datetime64[us]


Konvertierung in US/East und von da in UTC Timestamps

In [6]:
wti_v2 = wti.copy()

In [7]:
# 2. Den String in ein "naives" (zeitzonenloses) Datetime-Objekt umwandeln
# (Pandas erkennt das Format meistens automatisch)
wti_v2['time'] = pd.to_datetime(wti_v2['time'])

# 3. New York Zeit zuweisen (Lokalisieren)
# 'ambiguous='NaT'' fängt die doppelte Stunde bei der Umstellung im Herbst ab,
# 'nonexistent='NaT'' fängt die fehlende Stunde bei der Umstellung im Frühjahr ab.
wti_v2['timestamp_ny'] = wti_v2['time'].dt.tz_localize('America/New_York', ambiguous='NaT', nonexistent='NaT')

# 4. In UTC umrechnen
wti_v2['timestamp_utc'] = wti_v2['timestamp_ny'].dt.tz_convert('UTC')

# 5. (Optional) Wenn du das "+00:00" am Ende der UTC-Zeit nicht in der CSV haben willst,
# kannst du die Zeitzonen-Information wieder entfernen (die Zeit bleibt die von UTC):
# wti_v2['timestamp_utc_clean'] = wti_v2['timestamp_utc'].dt.tz_localize(None)

# Ergebnis überprüfen
#print(df[['Timestamp', 'Timestamp_UTC_clean']].head())

In [8]:
wti_v2.shape

(3883025, 7)

In [9]:
wti_v2 = (
    wti_v2
    .groupby('timestamp_utc', as_index=False)
    .agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last'
    })
)

In [10]:
wti_v2.shape

(3882805, 5)

In [11]:
wti_v2

,timestamp_utc,open,high,low,close
0,2011-01-03 01:15:00+00:00,91.280,91.290,91.260,91.260
1,2011-01-03 01:16:00+00:00,91.260,91.260,91.250,91.260
2,2011-01-03 01:17:00+00:00,91.250,91.260,91.250,91.260
3,2011-01-03 01:18:00+00:00,91.270,91.270,91.260,91.260
4,2011-01-03 01:19:00+00:00,91.250,91.250,91.250,91.250
...,...,...,...,...,...
3882800,2022-12-30 21:53:00+00:00,80.407,80.407,80.367,80.392
3882801,2022-12-30 21:54:00+00:00,80.402,80.427,80.387,80.422
3882802,2022-12-30 21:55:00+00:00,80.417,80.447,80.402,80.407
3882803,2022-12-30 21:56:00+00:00,80.427,80.427,80.369,80.387


In [12]:
wti_v2.dtypes

timestamp_utc    datetime64[us, UTC]
open                         float64
high                         float64
low                          float64
close                        float64
dtype: object

In [13]:
wti_v2.isna().sum()

timestamp_utc    0
open             0
high             0
low              0
close            0
dtype: int64

# Prüfen ob Minuten Intrday Daten vollständig sind

In [14]:
wti_v2["timestamp_utc"].dt.microsecond.value_counts().head()

timestamp_utc
0    3882805
Name: count, dtype: int64

In [15]:
wti_v2["timestamp_utc"].dt.second.value_counts().head()


timestamp_utc
0    3882805
Name: count, dtype: int64

In [16]:
full_range = pd.date_range(wti_v2["timestamp_utc"].min(), wti_v2["timestamp_utc"].max(), freq="min")
missing = len(full_range) - wti_v2["timestamp_utc"].nunique()
missing

2424198

In [17]:
wti_v2["timestamp_utc"].duplicated().any()

np.False_

es fehlen also massiv Minutendaten!

### Behebung durch Forward Fill und Flag

In [18]:
def prepare_wti_minute_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepares WTI intraday data for ML:
    - builds full 1-minute time index
    - forward fills OHLC
    - handles volume properly
    - creates missingness flags
    """

    # ----------------------------
    # 1. Datetime sauber setzen
    # ----------------------------
    df = df.copy()
    df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True)
    df = df.set_index('timestamp_utc').sort_index()

    # ----------------------------
    # 2. Full 1-minute index
    # ----------------------------
    full_index = pd.date_range(
        start=df.index.min(),
        end=df.index.max(),
        freq='1min',
        tz='UTC'
    )

    # ----------------------------
    # 3. Reindex (creates missing minutes)
    # ----------------------------
    df = df.reindex(full_index)

    # ----------------------------
    # 4. Missingness tracking BEFORE fill
    # ----------------------------
    df['was_missing'] = df['close'].isna().astype(int)

    # ----------------------------
    # 5. Forward fill OHLC
    # ----------------------------
    ohlc_cols = ['open', 'high', 'low', 'close']
    df[ohlc_cols] = df[ohlc_cols].ffill()

    # # ----------------------------
    # # 6. Volume handling
    # # ----------------------------
    # df['volume_missing'] = df['volume'].isna().astype(int)
    # df['volume'] = df['volume'].fillna(0)

    # # ----------------------------
    # # 7. Optional: market activity flag
    # # ----------------------------
    # df['is_active_market'] = (df['volume'] > 0).astype(int)

    # ----------------------------
    # 8. Sanity check (important!)
    # ----------------------------
    if df['close'].isna().any():
        raise ValueError("Forward fill failed - NaNs still present in close")

    return df

In [19]:
wti_v2 = prepare_wti_minute_data(wti_v2)

In [20]:
wti_v2 = wti_v2.reset_index()

In [21]:
wti_v2

,index,open,high,low,close,was_missing
0,2011-01-03 01:15:00+00:00,91.280,91.290,91.260,91.260,0
1,2011-01-03 01:16:00+00:00,91.260,91.260,91.250,91.260,0
2,2011-01-03 01:17:00+00:00,91.250,91.260,91.250,91.260,0
3,2011-01-03 01:18:00+00:00,91.270,91.270,91.260,91.260,0
4,2011-01-03 01:19:00+00:00,91.250,91.250,91.250,91.250,0
...,...,...,...,...,...,...
6306998,2022-12-30 21:53:00+00:00,80.407,80.407,80.367,80.392,0
6306999,2022-12-30 21:54:00+00:00,80.402,80.427,80.387,80.422,0
6307000,2022-12-30 21:55:00+00:00,80.417,80.447,80.402,80.407,0
6307001,2022-12-30 21:56:00+00:00,80.427,80.427,80.369,80.387,0


In [22]:
wti_v2 = wti_v2.rename(columns={"index": "timestamp_utc"})

In [25]:
wti_v2

,timestamp_utc,open,high,low,close,was_missing
0,2011-01-03 01:15:00+00:00,91.280,91.290,91.260,91.260,0
1,2011-01-03 01:16:00+00:00,91.260,91.260,91.250,91.260,0
2,2011-01-03 01:17:00+00:00,91.250,91.260,91.250,91.260,0
3,2011-01-03 01:18:00+00:00,91.270,91.270,91.260,91.260,0
4,2011-01-03 01:19:00+00:00,91.250,91.250,91.250,91.250,0
...,...,...,...,...,...,...
6306998,2022-12-30 21:53:00+00:00,80.407,80.407,80.367,80.392,0
6306999,2022-12-30 21:54:00+00:00,80.402,80.427,80.387,80.422,0
6307000,2022-12-30 21:55:00+00:00,80.417,80.447,80.402,80.407,0
6307001,2022-12-30 21:56:00+00:00,80.427,80.427,80.369,80.387,0


In [26]:
wti_v2["timestamp_utc"].is_monotonic_increasing

True

In [27]:
wti_v2.dtypes

timestamp_utc    datetime64[us, UTC]
open                         float64
high                         float64
low                          float64
close                        float64
was_missing                    int64
dtype: object

In [28]:
wti_v2.isna().sum()

timestamp_utc    0
open             0
high             0
low              0
close            0
was_missing      0
dtype: int64

In [29]:
# wti_v2.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\wti_v2.csv", sep=",", index=False, encoding="utf-8")

wti_v2.to_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\wti_v2.parquet", index=False)

In [30]:
wti_v3 = wti_v2.copy()

In [31]:
wti_v3.shape

(6307003, 6)

# Feature Engineering

Berechnung der Features auf basis von ffill datensatz, um die zeitlichen Abhängigkeiten nicht zu verzerren. Im Anschluss nur auf die echten Datenpunkte filtern, um die Ber

In [32]:
wti_v3 = build_wti_ohlc_features(wti_v3)

In [33]:
wti_v3 = create_labels(wti_v3)

In [36]:
wti_v3.columns

Index(['open', 'high', 'low', 'close', 'was_missing', 'log_close', 'ret_1',
       'hl_range', 'oc_return', 'ret_5', 'ret_15', 'ret_30', 'ret_60',
       'ret_240', 'mom_5', 'mom_15', 'mom_60', 'vol_close_5',
       'vol_parkinson_5', 'vol_close_15', 'vol_parkinson_15', 'vol_close_30',
       'vol_parkinson_30', 'vol_close_60', 'vol_parkinson_60', 'vol_close_240',
       'vol_parkinson_240', 'vol_gk', 'vol_ratio_5_60', 'sma_5', 'ema_5',
       'dist_ema_5', 'sma_15', 'ema_15', 'dist_ema_15', 'sma_60', 'ema_60',
       'dist_ema_60', 'sma_240', 'ema_240', 'dist_ema_240', 'hl_momentum',
       'body', 'upper_wick', 'lower_wick', 'abs_ret', 'vol_cluster', 'vol_5',
       'jump', 'hour', 'dayofweek', 'hour_sin', 'hour_cos', 'dow_sin',
       'dow_cos', 'us_session', 'vol_rank', 'vol_regime', 'trend_60',
       'trend_regime', 'y_up_2', 'y_down_2', 'y_up_5', 'y_down_5', 'y_up_15',
       'y_down_15', 'y_up_30', 'y_down_30', 'y_up_60', 'y_down_60', 'y_up_240',
       'y_down_240', 'y_up_720'

In [38]:
wti_v3.shape

(6307003, 76)

## Null Value Handling

In [39]:
na_counts = wti_v3.isna().sum()
na_counts = na_counts[na_counts > 0]

print(na_counts.to_string())

ret_1                  1
ret_5                  5
ret_15                15
ret_30                30
ret_60                60
ret_240              240
mom_5                  5
mom_15                15
mom_60                60
vol_close_5            5
vol_parkinson_5        4
vol_close_15          15
vol_parkinson_15      14
vol_close_30          30
vol_parkinson_30      29
vol_close_60          60
vol_parkinson_60      59
vol_close_240        240
vol_parkinson_240    239
vol_ratio_5_60        60
sma_5                  4
sma_15                14
sma_60                59
sma_240              239
hl_momentum            9
abs_ret                1
vol_cluster           10
vol_5                  5
vol_rank             560
trend_60              61


In [40]:
wti_v3 = wti_v3.reset_index()

In [41]:
wti_v3

,timestamp_utc,open,high,low,close,was_missing,log_close,ret_1,hl_range,oc_return,...,y_up_30,y_down_30,y_up_60,y_down_60,y_up_240,y_down_240,y_up_720,y_down_720,y_up_1440,y_down_1440
0,2011-01-03 01:15:00+00:00,91.280,91.290,91.260,91.260,0,4.513713,NaN,0.000329,-0.000219,...,0,0,0,0,0,0,0,0,0,0
1,2011-01-03 01:16:00+00:00,91.260,91.260,91.250,91.260,0,4.513713,0.000000,0.000110,0.000000,...,0,0,0,0,0,0,0,0,0,0
2,2011-01-03 01:17:00+00:00,91.250,91.260,91.250,91.260,0,4.513713,0.000000,0.000110,0.000110,...,0,0,0,0,0,0,0,0,0,0
3,2011-01-03 01:18:00+00:00,91.270,91.270,91.260,91.260,0,4.513713,0.000000,0.000110,-0.000110,...,0,0,0,0,0,0,0,0,0,0
4,2011-01-03 01:19:00+00:00,91.250,91.250,91.250,91.250,0,4.513603,-0.000110,0.000000,0.000000,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6306998,2022-12-30 21:53:00+00:00,80.407,80.407,80.367,80.392,0,4.386915,-0.000249,0.000498,-0.000187,...,0,0,0,0,0,0,0,0,0,0
6306999,2022-12-30 21:54:00+00:00,80.402,80.427,80.387,80.422,0,4.387288,0.000373,0.000497,0.000249,...,0,0,0,0,0,0,0,0,0,0
6307000,2022-12-30 21:55:00+00:00,80.417,80.447,80.402,80.407,0,4.387101,-0.000187,0.000560,-0.000124,...,0,0,0,0,0,0,0,0,0,0
6307001,2022-12-30 21:56:00+00:00,80.427,80.427,80.369,80.387,0,4.386852,-0.000249,0.000722,-0.000497,...,0,0,0,0,0,0,0,0,0,0


In [42]:
wti_v3["timestamp_utc"].is_monotonic_increasing

True

In [43]:
wti_v3["timestamp_utc"] = pd.to_datetime(wti_v3["timestamp_utc"], utc=True)

wti_v3 = wti_v3[wti_v3["timestamp_utc"] >= "2015-01-01"]

In [44]:
wti_v3.shape

(4206118, 77)

In [45]:
na_counts = wti_v3.isna().sum()
na_counts = na_counts[na_counts > 0]

print(na_counts.to_string())

Series([], )


In [ ]:
# wti_v3.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\wti_v3.csv", sep=",", index=False, encoding="utf-8")

In [46]:
wti_v3.to_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\wti_v3.parquet", index=False)

In [48]:
wti_v3

,timestamp_utc,open,high,low,close,was_missing,log_close,ret_1,hl_range,oc_return,...,y_up_30,y_down_30,y_up_60,y_down_60,y_up_240,y_down_240,y_up_720,y_down_720,y_up_1440,y_down_1440
2100885,2015-01-01 00:00:00+00:00,53.720,53.720,53.720,53.720,1,3.983785,0.000000,0.000000,0.000000,...,0,0,0,0,0,0,0,0,1,0
2100886,2015-01-01 00:01:00+00:00,53.720,53.720,53.720,53.720,1,3.983785,0.000000,0.000000,0.000000,...,0,0,0,0,0,0,0,0,1,0
2100887,2015-01-01 00:02:00+00:00,53.720,53.720,53.720,53.720,1,3.983785,0.000000,0.000000,0.000000,...,0,0,0,0,0,0,0,0,1,0
2100888,2015-01-01 00:03:00+00:00,53.720,53.720,53.720,53.720,1,3.983785,0.000000,0.000000,0.000000,...,0,0,0,0,0,0,0,0,1,0
2100889,2015-01-01 00:04:00+00:00,53.720,53.720,53.720,53.720,1,3.983785,0.000000,0.000000,0.000000,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6306998,2022-12-30 21:53:00+00:00,80.407,80.407,80.367,80.392,0,4.386915,-0.000249,0.000498,-0.000187,...,0,0,0,0,0,0,0,0,0,0
6306999,2022-12-30 21:54:00+00:00,80.402,80.427,80.387,80.422,0,4.387288,0.000373,0.000497,0.000249,...,0,0,0,0,0,0,0,0,0,0
6307000,2022-12-30 21:55:00+00:00,80.417,80.447,80.402,80.407,0,4.387101,-0.000187,0.000560,-0.000124,...,0,0,0,0,0,0,0,0,0,0
6307001,2022-12-30 21:56:00+00:00,80.427,80.427,80.369,80.387,0,4.386852,-0.000249,0.000722,-0.000497,...,0,0,0,0,0,0,0,0,0,0


## Klassenverteilung der Target Variablen

In [49]:
# import pandas as pd

def show_class_distribution(df, horizons=[2,5,15,30,60,240,720,1440]):
    results = []

    for h in horizons:
        up_col = f"y_up_{h}"
        down_col = f"y_down_{h}"

        if up_col in df.columns:
            up_counts = df[up_col].value_counts(normalize=True).sort_index()
            results.append({
                "horizon": h,
                "target": up_col,
                "class_0": up_counts.get(0, 0.0),
                "class_1": up_counts.get(1, 0.0)
            })

        if down_col in df.columns:
            down_counts = df[down_col].value_counts(normalize=True).sort_index()
            results.append({
                "horizon": h,
                "target": down_col,
                "class_0": down_counts.get(0, 0.0),
                "class_1": down_counts.get(1, 0.0)
            })

    result_df = pd.DataFrame(results)
    return result_df


# Nutzung:
dist = show_class_distribution(wti_v3)
print(dist)

    horizon       target   class_0   class_1
0         2       y_up_2  0.801505  0.198495
1         2     y_down_2  0.802617  0.197383
2         5       y_up_5  0.756548  0.243452
3         5     y_down_5  0.759272  0.240728
4        15      y_up_15  0.714431  0.285569
5        15    y_down_15  0.720045  0.279955
6        30      y_up_30  0.693491  0.306509
7        30    y_down_30  0.701634  0.298366
8        60      y_up_60  0.675478  0.324522
9        60    y_down_60  0.684706  0.315294
10      240     y_up_240  0.645163  0.354837
11      240   y_down_240  0.661728  0.338272
12      720     y_up_720  0.611595  0.388405
13      720   y_down_720  0.633382  0.366618
14     1440    y_up_1440  0.571126  0.428874
15     1440  y_down_1440  0.598143  0.401857


In [50]:
dist = show_class_distribution(wti_v3)
dist[["class_0", "class_1"]] = dist[["class_0", "class_1"]] * 100
print(dist.round(2))

    horizon       target  class_0  class_1
0         2       y_up_2    80.15    19.85
1         2     y_down_2    80.26    19.74
2         5       y_up_5    75.65    24.35
3         5     y_down_5    75.93    24.07
4        15      y_up_15    71.44    28.56
5        15    y_down_15    72.00    28.00
6        30      y_up_30    69.35    30.65
7        30    y_down_30    70.16    29.84
8        60      y_up_60    67.55    32.45
9        60    y_down_60    68.47    31.53
10      240     y_up_240    64.52    35.48
11      240   y_down_240    66.17    33.83
12      720     y_up_720    61.16    38.84
13      720   y_down_720    63.34    36.66
14     1440    y_up_1440    57.11    42.89
15     1440  y_down_1440    59.81    40.19


Gemini Ansatz für Triple Barrier Labeling

In [34]:
Auch hier muss man purgen - man nimmt dann den maximalen horizont der definiert ist!

SyntaxError: invalid syntax (3534078153.py, line 1)

## Die folgende Triple Barrier Funktion kann ignoriert werden

In [ ]:
import pandas as pd
import numpy as np

def apply_event_triple_barrier(df_prices, tweet_timestamps, x_minutes, pt_multiplier, sl_multiplier):
    """
    df_prices: DataFrame mit minütlichen WTI Preisen (Index: UTC Timestamp)
    tweet_timestamps: Liste oder Series von UTC Timestamps der Tweets
    x_minutes: Die vertikale Barriere (z.B. 60 für 1 Stunde)
    """
    labels = []
    
    for t in tweet_timestamps:
        # 1. Finde den exakten oder nächsten Preis-Zeitstempel zum Tweet
        if t not in df_prices.index:
            # Falls Markt geschlossen (Wochenende/Nacht), Event überspringen oder anpassen
            continue 
            
        p_t = df_prices.loc[t, 'close']
        
        # Volatilität (z.B. rollierende Standardabweichung) zur dynamischen Barrieren-Berechnung
        # Für ein einfaches Baseline-Modell tut es oft auch ein fixer Prozentsatz (z.B. 0.5%)
        volatility = df_prices.loc[t, 'daily_vol'] 
        
        upper_barrier = p_t * (1 + pt_multiplier * volatility)
        lower_barrier = p_t * (1 - sl_multiplier * volatility)
        vertical_barrier = t + pd.Timedelta(minutes=x_minutes)
        
        # Schaue in die Zukunft (Fenster von t bis t + x_minutes)
        future_prices = df_prices.loc[t:vertical_barrier, 'close']
        
        # Prüfe, welche Barriere zuerst getroffen wird
        first_touch_upper = future_prices[future_prices >= upper_barrier].index.min()
        first_touch_lower = future_prices[future_prices <= lower_barrier].index.min()
        
        # NaNs durch unendlich ersetzen für den Vergleich
        ft_upper = first_touch_upper if pd.notna(first_touch_upper) else pd.Timestamp.max
        ft_lower = first_touch_lower if pd.notna(first_touch_lower) else pd.Timestamp.max
        
        # Labeling Logik (Dein Ziel: 1 wenn es steigt, 0 wenn nicht)
        if ft_upper < ft_lower and ft_upper <= vertical_barrier:
            labels.append({'timestamp': t, 'label': 1}) # Profit-Target zuerst getroffen
        elif ft_lower < ft_upper and ft_lower <= vertical_barrier:
            labels.append({'timestamp': t, 'label': 0}) # Stop-Loss zuerst getroffen
        else:
            # Vertikale Barriere zuerst erreicht (Ablauf der Zeit x)
            # Hier entscheidest du: Ist der Preis nach x Minuten höher als am Anfang?
            final_price = future_prices.iloc[-1]
            label = 1 if final_price > p_t else 0
            labels.append({'timestamp': t, 'label': label})
            
    return pd.DataFrame(labels)